# Нейронные сети. Архитектуры нейронных сетей

## Подготовка данных

В этом практическом заданий мы будем решать задачу классификации цифр на датасете `mnist` с помощью полносвязной и сверточной нейронной сети. Для этого мы будем использовать надстройку над `tensorflow`, которая называется `keras`. Для начала обсудим данные. `mnist` датасет состоит из черно-белых изображений цифр размером $28 \times 28$ пикселей. В данном случае, мы работаем с одним каналом, хотя в случае цветных изображений, общее число каналов равно трем. Загрузим наши данные используя функцию `load_data` объекта `mnist` из модуля `keras.dataset`. Перед выполнением этого задания убедитесь, что ваша версия `tensorflow` >= 1.4.

In [1]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

In [2]:
tf.__version__

'1.15.0'

Нормализуйте заруженные данные `x_train` и `x_test`. Для этого следует разделить числовое значение каждого пикселя на $255$. Далее, переведите `y_train` и `y_test` в one-hot представление, используя функцию `tf.keras.utils.to_categorical`. Наше первое задание будет заключатся в реализации полносвязной нейронной сети. Поэтому измените размерность тренировочных и тестовых данных с помощью метода `reshape`.
    >> np_vector.shape
    >> (28, 28)
    >> np_vector = np_vector.reshape(28 * 28)
    >> np_vector.shape
    >> (784,)

### *РЕШЕНИЕ*

In [3]:
# Нормализация данных
x_train = x_train.reshape(60000, 28*28)
x_test = x_test.reshape(10000, 28*28)
x_train = x_train/255
x_test = x_test/255
# Переводим в one-hot
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)

## Полносвязная нейронная сеть.

В этой части задания вам предлагается реализовать обычную нейронную сеть с использованием последовательной модели `tf.keras.models.Sequential`. Данная модель позволяет добавлять слои при помощи функции встроенной `add`. Наша нейронная сеть будет состоять всего лишь из одного скрытого слоя с количеством нейроннов равным $256$, функцией активации ReLU и с `input_shape=(784,)`, что соответствует количеству нейронов во входном слое нашей нейронной сети. Количество нейроннов в выходном слое равно количеству классов, в качестве функции активации нужно использовать softmax. Не забудьте вызвать `model.compile` после добавления слоев. Используйте в качестве функции потерь `categorical_crossentropy`, оптимизатор `adadelta` и метрику `accuracy`.

### *РЕШЕНИЕ*

In [4]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(256, activation='relu', input_shape=(784,)))
model.add(tf.keras.layers.Dense(10, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adadelta',
    metrics=['accuracy']
)

Instructions for updating:
If using Keras pass *_constraint arguments to layers.


После этого, создайте модель и загрузите веса нейронной сети из файла `neural_networks.h5`. Какое количество настраиваемых параметров содержится в этой нейронной сети. Запишите это число в качестве первого ответа `answer1` на это задание. Оцените качетво на тестовой выборке и запишите это значение с точностью до трех знаков после запятой в переменную `answer2`.

### *РЕШЕНИЕ*

In [5]:
model.load_weights('datas/neural networks/neural_networks.h5')
model.fit(
    x = x_train,
    y = y_train,
    validation_data = (x_test, y_test)
)

Train on 60000 samples, validate on 10000 samples
60000/60000 [==============================] - 13s 208us/sample - loss: 0.0127 - acc: 0.9970 - val_loss: 0.0656 - val_acc: 0.9819


In [6]:
_, accuracy = model.evaluate(x_test, y_test)
model.summary(), \
accuracy

10000/10000 [==============================] - 1s 87us/sample - loss: 0.0656 - acc: 0.9819
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense (Dense)                (None, 256)               200960    
_________________________________________________________________
dense_1 (Dense)              (None, 10)                2570      
Total params: 203,530
Trainable params: 203,530
Non-trainable params: 0
_________________________________________________________________


(None, 0.9819)

In [7]:
answer1 = 203530
answer2 = accuracy

## Сверточная нейронная сеть

Далее, вам предлагается реализовать сверточную нейронную сеть. 

* Размерность входного слоя $(28, 28, 1)$.
* Сверточный слой с $32$ каналами, ядро свертки $3 \times 3$.
* Макспулинг слой $(2,2)$.
* Сверточный слой с $64$ каналами, ядро свертки $3 \times 3$.
* Макспулинг слой $(2,2)$.
* Понижение размерности признаков.
* Полносвязный слой с 64 нейронами
* Выходной слой с количеством нейронов равному количеству классов.

Для этого предлагается использовать следующие классы `Convolution2D`, `MaxPooling2D` и `Flatten` для понижения размерности признаков. Все эти классы как и слой полносвязной нейронной сети `Dense` находятся в `tf.keras.layers`. Используйте ReLU в качестве функции активации во всех слоях, где это потребуется, кроме выходного, в нем по аналогии с прошлым заданием используется softmax. Аналогичная ситуация с функцией `compile` после добавления слоев.

### *РЕШЕНИЕ*

In [8]:
model_s = tf.keras.Sequential([
    # Размерность входного слоя  (28,28,1)
    # Сверточный слой с  32  каналами, ядро свертки  3×3
    tf.keras.layers.Convolution2D(32, (3,3), activation='relu', input_shape=(28, 28, 1)),
    # Макспулинг слой  (2,2)
    tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
    # Сверточный слой с  64  каналами, ядро свертки  3×3
    tf.keras.layers.Convolution2D(64, (3,3), activation='relu'),
    # Макспулинг слой  (2,2)
    tf.keras.layers.MaxPooling2D(pool_size = (2,2)),
    # Понижение размерности признаков ???
    tf.keras.layers.Flatten(),
    # Полносвязный слой с 64 нейронами
    tf.keras.layers.Dense(64, activation='relu'),
    # Выходной слой с количеством нейронов равному количеству классов
    tf.keras.layers.Dense(10, activation='softmax')
])

model_s.compile(
    loss='categorical_crossentropy',
    optimizer='adadelta',
    metrics=['accuracy']
)

Теперь оцените качество получившейся модели на тестовой выборке. Для этого измените размерность `x_train` и `x_test` на размерность входного слоя. Загрузите веса `conv_net.h5`. Запишите количество параметров этой сверточной нейронной сети в `answer3`. Сравните его с количеством параметром в полносвязной нейронной сети, которую мы реализовали ранее. Оценку качества запишите в `answer4` с точностью до 3 трех знаков после запятой.

### *РЕШЕНИЕ*

In [9]:
x_train = x_train.reshape(60000, 28, 28, 1)
x_test = x_test.reshape(10000, 28, 28, 1)

In [ ]:
model_s.load_weights('datas/neural networks/conv_net.h5')

model_s.fit(
    x = x_train,
    y = y_train,
    validation_data = (x_test, y_test)
)

Train on 60000 samples, validate on 10000 samples
32128/60000 [===============>..............] - ETA: 47s - loss: 0.0022 - acc: 0.99

In [ ]:
_, accuracy = model_s.evaluate(x_test, y_test)
model_s.summary(), \
accuracy

In [ ]:
answer3 = 121930
answer4 = accuracy

# Строка с ответами

In [ ]:
output = "nn params {0}\n nn score {1:.3f}\ncnn params {2}\ncnn score {3:.3f}"
print(output.format(answer1, answer2, answer3, answer4))